In [2]:
# ============================================================
# CodeReviewer AI — Master Colab Notebook
# ============================================================
# This file is the single notebook that runs the entire pipeline
# in Google Colab. Each section marked with "# %%"  is one cell.
#
# How to use:
#   1. Upload this repo to Colab (or clone from GitHub)
#   2. Set Runtime → Change runtime type → T4 GPU
#   3. Run cells top to bottom
#   4. Estimated total runtime on free Colab T4: ~2-3 hours
#
# Colab hardware guide:
#   Free  T4  (15GB) → MODEL = "deepseek-ai/deepseek-coder-1.3b-instruct"
#   Pro   T4  (15GB) → MODEL = "deepseek-ai/deepseek-coder-6.7b-instruct"
#   Pro+  A100(40GB) → MODEL = "codellama/CodeLlama-13b-Instruct-hf"
# ============================================================

# %% [markdown]
# ## ⚙️ Cell 1 — Install all dependencies
# Run this first. It installs everything needed for all 5 stages.
# Takes ~3 minutes on first run, cached on subsequent runs.

# %%
# Cell 1: Install dependencies
import subprocess
packages = [
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "trl>=0.8.6",
    "bitsandbytes>=0.43.0",
    "datasets>=2.18.0",
    "accelerate>=0.28.0",
    "evaluate>=0.4.0",
    "rouge_score",
    "gradio>=4.0.0",
    "wandb",
    "tabulate",
    "sentencepiece",
]
subprocess.run(
    ["pip", "install", "--quiet"] + packages,
    check=True,
)
print("✅ All packages installed.")


✅ All packages installed.


In [3]:
# %% [markdown]
# ## 🔧 Cell 2 — Global config
# Change MODEL_NAME here to match your GPU tier.

# %%
# Cell 2: Global config — CHANGE MODEL_NAME TO MATCH YOUR GPU
import os, torch

# ── Pick ONE based on your Colab tier ──────────────────────────────────────
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"   # Free T4
# MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"  # Pro T4
# MODEL_NAME = "codellama/CodeLlama-13b-Instruct-hf"        # Pro+ A100

# Paths
DATA_DIR     = "data/processed"
SFT_CKPT     = "sft/checkpoints/final_adapter"
DPO_CKPT     = "dpo/checkpoints/final_adapter"
GRPO_CKPT    = "grpo/checkpoints/final_adapter"

for d in [DATA_DIR, "sft/checkpoints", "dpo/checkpoints", "grpo/checkpoints", "eval"]:
    os.makedirs(d, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Make sure you selected T4 runtime.")


Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# %% [markdown]
# ## 📦 Stage 1 — Data Pipeline
# ### Cell 3: Download CodeAlpaca (SFT data)

# %%
# Cell 3: Fetch CodeAlpaca dataset for SFT
import json
from datasets import load_dataset
from tqdm.auto import tqdm

SYSTEM_PROMPT = (
    "You are a senior software engineer performing a thorough code review. "
    "Identify bugs, inefficiencies, and style issues. Suggest specific improvements "
    "with clear explanations. Be concise but precise."
)

MAX_SFT_SAMPLES = 3000   # reduce to 1000 if you're on free tier and want faster training

print("Downloading CodeAlpaca-20k...")
ds = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
print(f"  Total available: {len(ds)}")

sft_file = os.path.join(DATA_DIR, "sft_codealpaca.jsonl")
written = 0
with open(sft_file, "w") as f:
    for ex in tqdm(ds, desc="Processing"):
        if written >= MAX_SFT_SAMPLES:
            break
        if not ex["input"].strip():
            continue
        instruction = ex["instruction"].strip()
        inp         = ex["input"].strip()
        output      = ex["output"].strip()
        user_msg    = f"{instruction}\n\n```\n{inp}\n```"
        record = {
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": user_msg},
                {"role": "assistant", "content": output},
            ]
        }
        f.write(json.dumps(record) + "\n")
        written += 1

print(f"✅ {written} SFT examples saved to {sft_file}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json:   0%|          | 0.00/8.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

  Total available: 20022


Processing:   0%|          | 0/20022 [00:00<?, ?it/s]

✅ 3000 SFT examples saved to data/processed/sft_codealpaca.jsonl


In [6]:
# %% [markdown]
# ### Cell 4: Build DPO preference pairs

# %%
# Cell 4: Build DPO pairs (LeetCode complexity heuristics + degraded SFT)
import re
from collections import defaultdict

DPO_FILE = os.path.join(DATA_DIR, "dpo_pairs.jsonl")
pairs = []

# ── Source A: LeetCode ────────────────────────────────────────────────────────
print("Trying to load LeetCode dataset...")
try:
    lc_ds = load_dataset("greengerong/leetcode", split="train")
    print(f"  LeetCode: {len(lc_ds)} examples")

    def has_hashmap(code):
        return bool(re.search(r'\bdict\b|\bset\b|\{\}|\bdefaultdict\b', code))

    def has_nested_loop(code):
        return bool(re.search(r'for .+ in .+:\n\s+(for|while)', code))

    def complexity_score(code):
        s = 0
        if has_hashmap(code): s += 2
        if not has_nested_loop(code): s += 2
        if "return" in code: s += 1
        if len(code.split("\n")) < 20: s += 1
        return s

    by_problem = defaultdict(list)
    for ex in lc_ds:
      title = ex.get("title", "").strip()
      code  = ex.get("python", ex.get("solution", "")).strip()
      if title and code:
        by_problem[title].append(code)

    for title, solutions in list(by_problem.items())[:500]:
      ranked   = sorted(solutions, key=complexity_score, reverse=True)
      chosen   = ranked[0]
      rejected = ranked[-1]

      prompt = (
            f"Review this solution for the LeetCode problem '{title}':\n\n"
            f"```python\n{chosen}\n```"
        )
      pairs.append({
            "prompt":   prompt,
            "chosen":   (
                "This solution uses an efficient approach:\n\n"
                " Leverages hash map for O(n) average-case lookup\n"
                " Avoids nested loops (no O(n²) behavior)\n"
                " Clean and readable implementation\n\n"
                "The time complexity is optimal for this problem."
            ),
            "rejected": (
                "The solution works. "
                "Consider reviewing the time complexity for potential improvements."
            ),
        })
    print(f"  Source A (LeetCode): {len(pairs)} pairs")

except Exception as e:
    print(f"  LeetCode dataset unavailable ({e}) — using Source B only.")

# ── Source B: Degrade SFT responses ──────────────────────────────────────────
def degrade(good: str) -> str:
    first_sentence = good.split(".")[0].strip() + "."
    return first_sentence + " The code could be improved. Consider refactoring for clarity."

source_b = []
with open(sft_file) as f:
    for line in f:
        if len(source_b) >= 500:
            break
        try:
            ex   = json.loads(line)
            msgs = ex["messages"]
            user = next(m["content"] for m in msgs if m["role"] == "user")
            good = next(m["content"] for m in msgs if m["role"] == "assistant")
            source_b.append({
                "prompt":   user,
                "chosen":   good,
                "rejected": degrade(good),
            })
        except (StopIteration, KeyError, json.JSONDecodeError):
            continue

print(f"  Source B (degraded SFT): {len(source_b)} pairs")
all_pairs = pairs + source_b
print(f"  Total DPO pairs: {len(all_pairs)}")

with open(DPO_FILE, "w") as f:
    for p in all_pairs:
        f.write(json.dumps(p) + "\n")

print(f"✅ DPO pairs saved to {DPO_FILE}")


Trying to load LeetCode dataset...
  LeetCode: 2360 examples
  Source A (LeetCode): 500 pairs
  Source B (degraded SFT): 500 pairs
  Total DPO pairs: 1000
✅ DPO pairs saved to data/processed/dpo_pairs.jsonl


In [7]:
# %% [markdown]
# ## 🎓 Stage 2 — SFT Training with QLoRA
# This is where we teach the model HOW to do code reviews.
# Expected time: ~45-60 min on free T4 with 1.3B model.

# %%
# Cell 5: SFT Training
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Load data
sft_examples = []
with open(sft_file) as f:
    for line in f:
        try:
            sft_examples.append(json.loads(line))
        except json.JSONDecodeError:
            continue

sft_dataset = Dataset.from_list(sft_examples)
print(f"SFT training examples: {len(sft_dataset)}")

# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer + model
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Attach LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Format chat
def format_chat(ex):
    text = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

sft_dataset = sft_dataset.map(format_chat, remove_columns=["messages"])

sft_args = SFTConfig(
    output_dir="sft/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none",

    # Dataset processing
    dataset_text_field="text",
    max_length=1024,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
)

print("🚀 Starting SFT training...")
trainer.train()
trainer.model.save_pretrained(SFT_CKPT)
tokenizer.save_pretrained(SFT_CKPT)
print(f"✅ SFT complete. Adapter saved to {SFT_CKPT}")

# Free memory
del model, trainer
torch.cuda.empty_cache()



SFT training examples: 3000
Loading deepseek-ai/deepseek-coder-1.3b-instruct...


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32014}.


🚀 Starting SFT training...


Step,Training Loss
10,1.727303
20,0.730092
30,0.504197
40,0.490304
50,0.458720
60,0.426646
70,0.444665
80,0.437801
90,0.427447
100,0.438457


✅ SFT complete. Adapter saved to sft/checkpoints/final_adapter


In [8]:
# %% [markdown]
# ## 🎯 Stage 3 — DPO: Teaching Preferences
# Now we teach the model WHAT a good review looks like.
# Expected time: ~30-45 min on free T4.

# %%
# Cell 6: DPO Training
from peft import PeftModel
from trl import DPOTrainer, DPOConfig

# Load DPO data
dpo_pairs = []
with open(DPO_FILE) as f:
    for line in f:
        try:
            ex = json.loads(line)
            if all(k in ex for k in ("prompt", "chosen", "rejected")):
                dpo_pairs.append(ex)
        except json.JSONDecodeError:
            continue

dpo_dataset = Dataset.from_list(dpo_pairs)
print(f"DPO pairs: {len(dpo_dataset)}")

# Load model + SFT adapter
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading model + SFT adapter...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(SFT_CKPT, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = PeftModel.from_pretrained(model, SFT_CKPT, is_trainable=True)

# DPO config
dpo_config = DPOConfig(
    output_dir="dpo/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    beta=0.1,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("🚀 Starting DPO training...")
dpo_trainer.train()
dpo_trainer.model.save_pretrained(DPO_CKPT)
tokenizer.save_pretrained(DPO_CKPT)
print(f"✅ DPO complete. Adapter saved to {DPO_CKPT}")

del model, dpo_trainer
torch.cuda.empty_cache()



DPO pairs: 1000
Loading model + SFT adapter...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32014}.


🚀 Starting DPO training...


Step,Training Loss
10,0.164618
20,0.001582
30,0.000195
40,0.000189
50,0.000350
60,0.000142
70,0.000153
80,0.000134
90,0.000190
100,0.000162


✅ DPO complete. Adapter saved to dpo/checkpoints/final_adapter


In [16]:
!zip -r my_project_backup.zip codereviewer-ai/ data/ dpo/ eval/ grpo/ sft/

  adding: codereviewer-ai/ (stored 0%)
  adding: data/ (stored 0%)
  adding: data/processed/ (stored 0%)
  adding: data/processed/sft_codealpaca.jsonl (deflated 83%)
  adding: data/processed/dpo_pairs.jsonl (deflated 79%)
  adding: dpo/ (stored 0%)
  adding: dpo/checkpoints/ (stored 0%)
  adding: dpo/checkpoints/checkpoint-125/ (stored 0%)
  adding: dpo/checkpoints/checkpoint-125/rng_state.pth (deflated 26%)
  adding: dpo/checkpoints/checkpoint-125/tokenizer_config.json (deflated 40%)
  adding: dpo/checkpoints/checkpoint-125/trainer_state.json (deflated 73%)
  adding: dpo/checkpoints/checkpoint-125/tokenizer.json (deflated 81%)
  adding: dpo/checkpoints/checkpoint-125/optimizer.pt (deflated 21%)
  adding: dpo/checkpoints/checkpoint-125/chat_template.jinja (deflated 57%)
  adding: dpo/checkpoints/checkpoint-125/adapter_config.json (deflated 60%)
  adding: dpo/checkpoints/checkpoint-125/training_args.bin (deflated 53%)
  adding: dpo/checkpoints/checkpoint-125/ref/ (stored 0%)
  adding: d

In [17]:
!ls -lh my_project_backup.zip

-rw-r--r-- 1 root root 551M Jun  3 22:57 my_project_backup.zip


In [18]:
from google.colab import files
files.download('my_project_backup.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# %% [markdown]
# ## 🧠 Stage 4 — GRPO: Group Preference Training (Optional)
# Most advanced stage — the same technique used in DeepSeek-R1.
# Skip this on free Colab if short on time; it's a bonus.

# %%
# Cell 7: GRPO Training (optional — skip on free T4 if needed)
from trl import GRPOTrainer, GRPOConfig

ISSUE_KW = ["bug","error","inefficien","slow","o(n","time complexity",
            "memory","security","injection","overflow","null","exception"]
FIX_KW   = ["instead","replace","use ","consider ","try ","better to",
            "recommend","suggest","change","rewrite","refactor"]
WHY_KW   = ["because","since","this causes","this means","which leads",
            "result in","due to","as a result"]

def compute_reward(text):
    r, lower = 0.0, text.lower()
    if any(k in lower for k in ISSUE_KW): r += 2.0
    if any(k in lower for k in FIX_KW):   r += 2.0
    if any(k in lower for k in WHY_KW):    r += 1.0
    wc = len(text.split())
    if 30 <= wc <= 300: r += 1.0
    elif wc < 10:        r -= 1.0
    return r

def batch_reward_fn(completions, **kwargs):
    results = []
    for c in completions:
        text = "".join(x.get("content","") for x in c) if isinstance(c, list) else str(c)
        results.append(compute_reward(text))
    return results

# Load prompts
grpo_prompts = []
with open(sft_file) as f:
    for line in f:
        if len(grpo_prompts) >= 300:
            break
        try:
            ex   = json.loads(line)
            msgs = ex["messages"]
            user = next(m["content"] for m in msgs if m["role"] == "user")
            grpo_prompts.append({"prompt": user})
        except (StopIteration, KeyError, json.JSONDecodeError):
            continue

grpo_dataset = Dataset.from_list(grpo_prompts)
print(f"GRPO prompts: {len(grpo_dataset)}")

# Load model + DPO adapter
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(DPO_CKPT, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = PeftModel.from_pretrained(model, DPO_CKPT, is_trainable=True)

grpo_config = GRPOConfig(
    output_dir="grpo/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_generations=4,
    max_completion_length=200,
    bf16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    max_steps=150,
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    reward_funcs=batch_reward_fn,
    processing_class=tokenizer,
)

print("🚀 Starting GRPO training (150 steps)...")
grpo_trainer.train()
grpo_trainer.model.save_pretrained(GRPO_CKPT)
tokenizer.save_pretrained(GRPO_CKPT)
print(f"✅ GRPO complete. Adapter saved to {GRPO_CKPT}")

del model, grpo_trainer
torch.cuda.empty_cache()



GRPO prompts: 300


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32014}.


🚀 Starting GRPO training (150 steps)...


Step,Training Loss
10,-0.323996
20,-0.316878
30,-0.421939
40,-0.268493
50,-0.252560
60,-0.339171
70,-0.219144
80,-0.365189
90,-0.345532
100,-0.278426


✅ GRPO complete. Adapter saved to grpo/checkpoints/final_adapter


In [20]:
# %% [markdown]
# ## 📊 Stage 5a — Evaluation
# Compare all checkpoints on ROUGE-L and rule-based reward.

# %%
# Cell 8: Evaluation — compare all checkpoints
import evaluate as hf_evaluate

rouge = hf_evaluate.load("rouge")

def load_for_inference(adapter_path=None):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb,
        device_map="auto", trust_remote_code=True,
    )
    tok_src = adapter_path if (adapter_path and os.path.exists(adapter_path)) else MODEL_NAME
    t = AutoTokenizer.from_pretrained(tok_src, trust_remote_code=True)
    if t.pad_token is None:
        t.pad_token = t.eos_token
    if adapter_path and os.path.exists(adapter_path):
        m = PeftModel.from_pretrained(m, adapter_path)
    m.eval()
    return m, t

def generate(m, t, prompt, max_new=150):
    inp = t(
        f"Review this code:\n{prompt[:400]}\n\nReview:",
        return_tensors="pt", truncation=True, max_length=512,
    ).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_new, temperature=0.3,
            do_sample=True, pad_token_id=t.pad_token_id,
            eos_token_id=t.eos_token_id,
        )
    new = out[0][inp["input_ids"].shape[1]:]
    return t.decode(new, skip_special_tokens=True).strip()

# Load 30 eval samples
eval_samples = []
with open(sft_file) as f:
    for line in f:
        if len(eval_samples) >= 30:
            break
        try:
            ex   = json.loads(line)
            msgs = ex["messages"]
            user = next(m["content"] for m in msgs if m["role"] == "user")
            ref  = next(m["content"] for m in msgs if m["role"] == "assistant")
            eval_samples.append({"prompt": user, "reference": ref})
        except (StopIteration, KeyError, json.JSONDecodeError):
            continue

checkpoints_to_eval = [
    ("Base model",  None),
    ("After SFT",   SFT_CKPT),
    ("After DPO",   DPO_CKPT),
]
if os.path.exists(GRPO_CKPT):
    checkpoints_to_eval.append(("After GRPO", GRPO_CKPT))

results_table = []
for name, ckpt in checkpoints_to_eval:
    print(f"\nEvaluating {name}...")
    m, t = load_for_inference(ckpt)
    preds = [generate(m, t, s["prompt"]) for s in tqdm(eval_samples, desc=name)]
    refs  = [s["reference"] for s in eval_samples]
    rouge_score = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
    avg_reward  = sum(compute_reward(p) for p in preds) / len(preds)
    results_table.append({
        "Model":    name,
        "ROUGE-L":  f"{rouge_score['rougeL']:.4f}",
        "Reward":   f"{avg_reward:.4f}",
    })
    del m, t
    torch.cuda.empty_cache()

# Print results
print("\n" + "="*55)
print(f"{'Model':<25} {'ROUGE-L':>10} {'Reward':>10}")
print("-"*55)
for r in results_table:
    print(f"{r['Model']:<25} {r['ROUGE-L']:>10} {r['Reward']:>10}")
print("="*55)




Evaluating Base model...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Base model:   0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

After DPO:   0%|          | 0/30 [00:00<?, ?it/s]


Evaluating After GRPO...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

After GRPO:   0%|          | 0/30 [00:00<?, ?it/s]


Model                        ROUGE-L     Reward
-------------------------------------------------------
Base model                    0.2551     2.2333
After SFT                     0.3038     1.4000
After DPO                     0.2418     1.1333
After GRPO                    0.2743     1.3333


In [21]:
# %% [markdown]
# ## 🖥️ Stage 5b — Gradio Demo
# Launch the interactive UI. Colab will show a public link.

# %%
# Cell 9: Launch Gradio demo
import gradio as gr

# Find best adapter
best_label, best_ckpt = "Base model", None
for label, path in [("GRPO", GRPO_CKPT), ("DPO", DPO_CKPT), ("SFT", SFT_CKPT)]:
    if os.path.exists(path):
        best_label, best_ckpt = label, path
        break

print(f"Loading {best_label} for demo...")
demo_model, demo_tokenizer = load_for_inference(best_ckpt)

DEMO_SYSTEM = (
    "You are a senior software engineer performing a thorough code review. "
    "Identify bugs, inefficiencies, and style issues. Be concise but precise."
)

def review_code_demo(code, language, max_tokens, temperature):
    if not code.strip():
        return "⚠️ Please paste some code."
    messages = [
        {"role": "system", "content": DEMO_SYSTEM},
        {"role": "user",   "content": f"Review this {language} code:\n\n```{language.lower()}\n{code}\n```"},
    ]
    try:
        formatted = demo_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        formatted = f"System: {DEMO_SYSTEM}\n\nUser: Review:\n{code}\n\nAssistant:"

    inputs = demo_tokenizer(
        formatted, return_tensors="pt", truncation=True, max_length=1024
    ).to(demo_model.device)

    with torch.no_grad():
        out = demo_model.generate(
            **inputs, max_new_tokens=int(max_tokens),
            temperature=float(temperature), do_sample=temperature > 0.1,
            top_p=0.95, repetition_penalty=1.1,
            pad_token_id=demo_tokenizer.pad_token_id,
            eos_token_id=demo_tokenizer.eos_token_id,
        )
    new = out[0][inputs["input_ids"].shape[1]:]
    return demo_tokenizer.decode(new, skip_special_tokens=True).strip()

with gr.Blocks(title="CodeReviewer AI", theme=gr.themes.Soft()) as colab_demo:
    gr.Markdown(f"# 🔍 CodeReviewer AI  ({best_label})")
    with gr.Row():
        code_in  = gr.Textbox(label="Code", placeholder="Paste code here...", lines=12)
        code_out = gr.Textbox(label="Review", lines=12, interactive=False)
    lang    = gr.Dropdown(["Python","JavaScript","Java","TypeScript","Go"], value="Python", label="Language")
    max_tok = gr.Slider(50, 400, value=200, step=10, label="Max tokens")
    temp    = gr.Slider(0.1, 1.0, value=0.3, step=0.05, label="Temperature")
    gr.Button("🔍 Review", variant="primary").click(
        review_code_demo, [code_in, lang, max_tok, temp], code_out
    )
    gr.Examples(
        [["def find_max(arr):\n    return sorted(arr)[-1]", "Python", 200, 0.3]],
        inputs=[code_in, lang, max_tok, temp],
    )

# share=True gives a public URL in Colab
colab_demo.launch(share=True, show_error=True)

Loading GRPO for demo...


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/tmp/ipykernel_4231/4244255076.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="CodeReviewer AI", theme=gr.themes.Soft()) as colab_demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://acb2cfa073d25ad658.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
